# 03 - Auto-ARIMA: Selecao Automatica de Ordem

Este notebook demonstra a selecao automatica de ordem usando **auto_arima** do chronobox.

**Conteudo:**
1. Criterios de informacao (AIC, BIC, AICc)
2. Algoritmo stepwise de Hyndman-Khandakar
3. Auto-ARIMA com chronobox
4. Comparacao de criterios
5. Validacao cruzada temporal (rolling window)
6. Exercicios

## 1. Criterios de Informacao

A selecao de ordem busca equilibrar ajuste e parcimonia:

- **AIC** (Akaike): $\text{AIC} = -2\ell + 2k$ — minimiza perda de informacao
- **BIC** (Bayesian): $\text{BIC} = -2\ell + k \ln(n)$ — penaliza mais modelos complexos
- **AICc** (Corrected): $\text{AICc} = \text{AIC} + \frac{2k(k+1)}{n-k-1}$ — correcao para amostras finitas

Onde $\ell$ e a log-verossimilhanca, $k$ o numero de parametros e $n$ o numero de observacoes.

**Regra geral:** AICc e preferido para amostras pequenas/moderadas; BIC e consistente (seleciona o modelo verdadeiro se este estiver entre os candidatos).

## 2. Setup e Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chronobox import ARIMA, auto_arima
from chronobox.tests_stat import ljung_box_test
from chronobox.visualization import plot_diagnostics, plot_forecast, set_theme

set_theme('professional')
np.random.seed(42)

print('chronobox importado com sucesso!')

In [ ]:
import os
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

# Dataset airline (sazonal)
airline = pd.read_csv(os.path.join(DATA_DIR, 'airline.csv'), parse_dates=['date'])
airline.set_index('date', inplace=True)
y_airline = np.log(airline['passengers'].values)  # log para estabilizar variancia

# Dataset Nile (nao-sazonal)
nile = pd.read_csv(os.path.join(DATA_DIR, 'nile.csv'), parse_dates=['date'])
nile.set_index('date', inplace=True)
y_nile = nile['flow'].values

print(f'Airline: {len(y_airline)} obs (mensal, s=12)')
print(f'Nile: {len(y_nile)} obs (anual, sem sazonalidade)')

## 3. Auto-ARIMA: Serie Nao-Sazonal (Nile)

In [ ]:
# Auto-ARIMA stepwise (padrao) no Nile
best_nile = auto_arima(
    y_nile,
    seasonal=False,
    max_p=5, max_q=5,
    information_criterion='aicc',
    stepwise=True,
    trace=True  # mostra progresso
)

print()
print(best_nile.summary())

In [ ]:
# Grafico 1: Diagnostico do modelo selecionado automaticamente
fig = plot_diagnostics(best_nile, lags=15, title=f'Diagnostico - {best_nile.model_name} (Nile)')
plt.show()

## 4. Auto-ARIMA: Serie Sazonal (Airline)

In [ ]:
# Auto-ARIMA com sazonalidade no Airline
best_airline = auto_arima(
    y_airline,
    seasonal=True,
    m=12,
    max_p=3, max_q=3,
    max_P=2, max_Q=2,
    information_criterion='aicc',
    stepwise=True,
    trace=True
)

print()
print(best_airline.summary())

In [ ]:
# Grafico 2: Previsao do melhor modelo sazonal
fig = plot_forecast(best_airline, steps=24, alpha=0.05,
                    title=f'Previsao {best_airline.model_name} - Airline')
plt.show()

## 5. Comparacao de Criterios de Informacao

Vamos ajustar varios modelos manualmente e comparar com a selecao automatica.

In [ ]:
# Grid de modelos para comparacao
modelos_airline = [
    ((0,1,1), (0,1,1,12)),
    ((1,1,0), (0,1,1,12)),
    ((1,1,1), (0,1,1,12)),
    ((0,1,1), (1,1,1,12)),
    ((1,1,1), (1,1,1,12)),
    ((2,1,1), (0,1,1,12)),
    ((0,1,2), (0,1,1,12)),
]

resultados = []
for order, seasonal in modelos_airline:
    try:
        m = ARIMA(order=order, seasonal_order=seasonal)
        r = m.fit(y_airline)
        resultados.append({
            'Modelo': m.model_name,
            'AIC': r.aic,
            'BIC': r.bic,
            'AICc': r.aicc,
            'LogLik': r.loglike,
            'Params': len(r.params)
        })
    except Exception as e:
        print(f'Falha em {order}{seasonal}: {e}')

df_comp = pd.DataFrame(resultados)
df_comp = df_comp.sort_values('AICc')
print(df_comp.to_string(index=False))

print(f'\nMelhor por AICc: {df_comp.iloc[0]["Modelo"]}')
print(f'Melhor por BIC:  {df_comp.sort_values("BIC").iloc[0]["Modelo"]}')

In [ ]:
# Grafico 3: Comparacao visual dos criterios
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

modelos_nomes = df_comp['Modelo'].values
x = range(len(modelos_nomes))

for ax, criterio, cor in zip(axes, ['AIC', 'BIC', 'AICc'], ['steelblue', 'darkorange', 'green']):
    vals = df_comp[criterio].values
    bars = ax.bar(x, vals, color=cor, alpha=0.7)
    melhor_idx = np.argmin(vals)
    bars[melhor_idx].set_color('red')
    bars[melhor_idx].set_alpha(1.0)
    ax.set_title(criterio)
    ax.set_xticks(x)
    ax.set_xticklabels(modelos_nomes, rotation=45, ha='right', fontsize=8)

plt.suptitle('Comparacao de Criterios de Informacao (melhor em vermelho)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Validacao Cruzada Temporal (Rolling Window)

A validacao cruzada temporal avalia a capacidade preditiva real do modelo, evitando overfitting.

**Procedimento:**
1. Definir tamanho minimo de treino e horizonte de previsao
2. Para cada fold: treinar no passado, prever o futuro proximo
3. Calcular metricas de erro (RMSE, MAE) em cada fold
4. Reportar media e desvio das metricas

In [ ]:
# Implementacao de rolling window CV
def rolling_cv(y, order, seasonal_order, n_folds=5, horizon=12, min_train=60):
    """Validacao cruzada temporal com janela expansiva."""
    n = len(y)
    fold_size = (n - min_train - horizon) // n_folds
    
    results = []
    for i in range(n_folds):
        train_end = min_train + i * fold_size
        test_end = min(train_end + horizon, n)
        
        y_train = y[:train_end]
        y_test = y[train_end:test_end]
        
        if len(y_test) == 0:
            break
        
        model = ARIMA(order=order, seasonal_order=seasonal_order)
        fit = model.fit(y_train)
        fc = fit.forecast(steps=len(y_test))
        
        errors = y_test - fc['forecast']
        rmse = np.sqrt(np.mean(errors**2))
        mae = np.mean(np.abs(errors))
        
        results.append({
            'fold': i + 1,
            'train_size': len(y_train),
            'test_size': len(y_test),
            'rmse': rmse,
            'mae': mae
        })
    
    return pd.DataFrame(results)

print('Funcao rolling_cv definida.')

In [ ]:
# CV para varios modelos no airline
modelos_cv = [
    ('ARIMA(1,1,1)', (1,1,1), (0,0,0,0)),
    ('SARIMA(0,1,1)(0,1,1)[12]', (0,1,1), (0,1,1,12)),
    ('SARIMA(1,1,1)(0,1,1)[12]', (1,1,1), (0,1,1,12)),
    ('SARIMA(0,1,1)(1,1,1)[12]', (0,1,1), (1,1,1,12)),
]

cv_results = {}
for nome, order, seasonal in modelos_cv:
    print(f'Rodando CV para {nome}...')
    cv = rolling_cv(y_airline, order, seasonal, n_folds=5, horizon=12, min_train=72)
    cv_results[nome] = cv
    print(f'  RMSE medio: {cv["rmse"].mean():.4f} (+/- {cv["rmse"].std():.4f})')
    print(f'  MAE medio:  {cv["mae"].mean():.4f} (+/- {cv["mae"].std():.4f})')
    print()

In [ ]:
# Grafico 4: Comparacao CV - RMSE por modelo
fig, ax = plt.subplots(figsize=(10, 5))

nomes = list(cv_results.keys())
rmse_means = [cv_results[n]['rmse'].mean() for n in nomes]
rmse_stds = [cv_results[n]['rmse'].std() for n in nomes]

colors = ['#e74c3c' if v == min(rmse_means) else '#3498db' for v in rmse_means]
bars = ax.bar(range(len(nomes)), rmse_means, yerr=rmse_stds, 
              color=colors, capsize=5, alpha=0.8)
ax.set_xticks(range(len(nomes)))
ax.set_xticklabels(nomes, rotation=30, ha='right')
ax.set_ylabel('RMSE')
ax.set_title('Validacao Cruzada Temporal: RMSE por Modelo (melhor em vermelho)')
plt.tight_layout()
plt.show()

In [ ]:
# Grafico 5: Evolucao do erro por fold (rolling window)
fig, ax = plt.subplots(figsize=(10, 5))

for nome, cv in cv_results.items():
    ax.plot(cv['fold'], cv['rmse'], marker='o', label=nome, linewidth=2)

ax.set_xlabel('Fold')
ax.set_ylabel('RMSE')
ax.set_title('Evolucao do RMSE por Fold na CV Temporal')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('Nota: se o erro cresce nos folds finais, pode indicar mudanca estrutural na serie.')

## 7. Stepwise vs Grid Search

In [ ]:
import time

# Stepwise (rapido)
t0 = time.time()
best_stepwise = auto_arima(y_airline, seasonal=True, m=12, stepwise=True, 
                           information_criterion='aicc', trace=False)
t_stepwise = time.time() - t0

# Grid search (exaustivo - pode demorar)
t0 = time.time()
best_grid = auto_arima(y_airline, seasonal=True, m=12, stepwise=False, 
                       max_p=2, max_q=2, max_P=1, max_Q=1,
                       information_criterion='aicc', trace=False)
t_grid = time.time() - t0

print(f'Stepwise: {best_stepwise.model_name}, AICc={best_stepwise.aicc:.2f}, tempo={t_stepwise:.2f}s')
print(f'Grid:     {best_grid.model_name}, AICc={best_grid.aicc:.2f}, tempo={t_grid:.2f}s')
print(f'\nSpeedup stepwise: {t_grid/t_stepwise:.1f}x')
print('O stepwise geralmente encontra o mesmo modelo (ou muito proximo) em fracao do tempo.')

## Exercicio

### Exercicio 1: Auto vs Manual

1. Carregue o dataset `brazil_ipca.csv`
2. Rode `auto_arima` com `seasonal=True, m=12`
3. Escolha manualmente um modelo SARIMA baseado em ACF/PACF
4. Compare ambos via:
   - Criterios de informacao (AIC, BIC)
   - Validacao cruzada temporal (5 folds, horizonte 12)
5. Qual abordagem produziu melhor resultado preditivo?

In [ ]:
# Exercicio 1 - Seu codigo aqui
# ipca = pd.read_csv(os.path.join(DATA_DIR, 'brazil_ipca.csv'), parse_dates=['date'])
# ...

## Exercicio

### Exercicio 2: Sensibilidade ao criterio de informacao

1. Rode `auto_arima` no Nile com `information_criterion='aic'`
2. Rode novamente com `information_criterion='bic'`
3. Os modelos selecionados sao diferentes? Por que?
4. Qual criterio voce recomendaria para esta serie e por que?

In [ ]:
# Exercicio 2 - Seu codigo aqui
# ...